# WildSpeech M8.6 — M0++ Item-Response Intelligence & Competitive Screening

**Research objective:** improve the frozen M8.5 E04 candidate (student pooled OOF R²≈0.2126) using B1 full-feature fusion, B2 monotone additive theta model, item-response and Rasch features, instrument partial pooling, and clip-duration nuisance controls.

**Interpretation:** M8.4 reference is formally approved, M8.5 E04 has an exploratory HOLD_REFERENCE decision. The code has separate engineering submission and scientific claims. Reported public R²≈0.5 is NOT a local model score and must not be compared directly.

**Data safety:** Google Drive first, private student-level outputs, original frozen folds and OOF untouched, no external inference APIs, no audio reads, no test-data fitting. No online leaderboard submission.

The first run is SCREEN; change MODE to FULL for all 50 student plus 50 school-disjoint folds. FULL is computationally substantial and checkpointed.

## 0 · Mount private Google Drive

Expected existing folders: `MyDrive/WildSpeech/`, `WildSpeech_M84/phase0/`, `WildSpeech_M85/m0plus/`.

In [ ]:
import sys,os,json,subprocess,importlib,time
from pathlib import Path
from IPython.display import display,Markdown,Image
import pandas as pd
import numpy as np
try:
    IN_COLAB=importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB=False
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive",force_remount=False)
    DRIVE=Path("/content/drive/MyDrive")
else:
    DRIVE=Path(os.environ.get("WILDSPEECH_DRIVE_ROOT",str(Path.home()/"MyDrive")))
assert DRIVE.is_dir(),f"Private Drive not mounted at {DRIVE}"
print("Drive:",DRIVE)

## 1 · Install versioned GitHub package

The commit SHA is stored with every experiment. No ZIP upload or untracked direct notebook code is used.

In [ ]:
GITHUB="https://github.com/mybenkhadda/WildSpeech.git"
CHECKOUT=Path("/content/WildSpeech_git") if IN_COLAB else Path.cwd()/"WildSpeech_git"
if not (CHECKOUT/".git").is_dir():
    if CHECKOUT.exists():raise RuntimeError(f"Directory exists but is not a Git checkout: {CHECKOUT}")
    subprocess.run(["git","clone","--depth","1","--branch","main",GITHUB,str(CHECKOUT)],check=True)
else:
    subprocess.run(["git","-C",str(CHECKOUT),"pull","--ff-only","origin","main"],check=True)
PKG=CHECKOUT/"WildSpeech_M8"
subprocess.run([sys.executable,"-m","pip","install","-q","-e",str(PKG)],check=True)
src=str(PKG/"src")
if src not in sys.path:sys.path.insert(0,src)
from wildspeech_m86.experiments import execute,EXPERIMENTS,plan
from wildspeech_m86.contracts import FULL_FOLD_HASH
from wildspeech_m84.data import load_official,locate_data_root,digest_bytes
GIT_SHA=subprocess.check_output(["git","-C",str(CHECKOUT),"rev-parse","HEAD"],text=True).strip()
print("M8.6 source SHA:",GIT_SHA)

## 2 · Mode selection

SCREEN is exploratory and executes only the first five student + five school folds with a single lightweight CatBoost seed. FULL uses **all eleven preregistered models**, the full immutable 100-fold protocol, five CatBoost seeds and fixed hyperparameters.

The original Phase 0, M8.5 reports, and OOF predictions are READ ONLY. The outputs of SCREEN and FULL live in separate versioned subdirectories.

In [ ]:
MODE="SCREEN"                 # change to "FULL" only after initial diagnostics
EXPERIMENT_IDS=list(EXPERIMENTS)
DATA_ROOT_OVERRIDE=""
PHASE0_ROOT=DRIVE/"WildSpeech_M84"/"phase0"
M85_ROOT=DRIVE/"WildSpeech_M85"/"m0plus"
OUTPUT_ROOT=DRIVE/"WildSpeech_M86"/"m0pp"
EXPECTED_FOLD_HASH=FULL_FOLD_HASH
EXPORT_CANDIDATE=True        # only FULL, and only if engineering gate passes
EXPORT_E04_DIAGNOSTIC=False  # explicit opt-in; old HOLD_REFERENCE is preserved
assert MODE in ("SCREEN","FULL")
assert PHASE0_ROOT.joinpath("folds","folds.parquet").exists()
assert M85_ROOT.joinpath("reports","M85_RESULTS.json").exists()
print("Mode:",MODE)
print("Output:",OUTPUT_ROOT/MODE.lower())

## 3 · Freeze and verify E00 and E04 anchors

The experiment runner validates official source CSV SHA-256 values, metadata/label student order, the 100 stored folds, M8.4 reference OOF, E04 OOF checksum and completed M8.5 ledger before creating candidate runs.

In [ ]:
DATA_ROOT=locate_data_root(DRIVE,DATA_ROOT_OVERRIDE)
DATA=load_official(DATA_ROOT)
M85_HISTORY=json.loads((M85_ROOT/"reports"/"M85_RESULTS.json").read_text())
print("M8.5 mode:",M85_HISTORY["mode"],"decision:",M85_HISTORY["decision"])
print("Reference frozen mean-fold R²:",M85_HISTORY["reference"]["mean_fold_r2"])
e04=next(r for r in M85_HISTORY["model_metrics"] if r["model"]=="E04")
display(pd.DataFrame([{"model":"E04",**e04}]).T)
print("Locked fold hash:",M85_HISTORY["fold_hash"])
assert M85_HISTORY["fold_hash"]==FULL_FOLD_HASH
assert M85_HISTORY["mode"]=="FULL" and M85_HISTORY["completed_all_E01_E09"]
print("Train/test:",len(DATA.train_students),len(DATA.test_students))
print("Train/test clip rows:",len(DATA.train_clips),len(DATA.test_clips))

## 4 · Explicit experimental preregistration

**B1** Ridge/CatBoost fusion on full E04 features; **B2** constrained monotone theta GAM; **B3** S0+S2+S3; **I1–I5** item-response matrix, Rasch, item complexity, difficulty profiles and bundle; **D2/D3** partial pooling; **T1** item×score-normalized clip duration.

'First error', 'last three errors' and 'furthest reached' are **disabled** because the official schema does not provide verified item chronology.

In [ ]:
PLAN=pd.DataFrame(plan())
display(PLAN[["comparison_id","baseline","challenger","primary_science_test"]])
print("Comparisons:",len(PLAN))
print("Primary predefined science hypotheses: B1, B2, I5")
print("Engineering rule: positive pooled Δ, ≥65% wins, school Δ≥−.005, instrument Δ≥−.01")
print("Science rule: mean fold Δ≥+.01, corrected t≥2, ≥60% wins + same robustness vetoes")

## 5 · Review data protocol and item feature validity

In [ ]:
from wildspeech_m86.contracts import prompt_key
trainclips=DATA.train_clips
print("Observed tasks:",sorted(trainclips.task.unique()))
print("Has raw expected prompt text:","expected_text" in trainclips)
print("Item score available on hidden test:",
      ("item_score" in DATA.test_clips) and DATA.test_clips.item_score.notna().any())
print("No verified item order:",not any(x in trainclips for x in
     ("item_order","item_sequence","timestamp","start_time")))
print("Features do NOT include future outcome, score_status, school, child ID or filename.")

## 6 · Execute M8.6 screening/full experiment engine

The heavy work is performed by the library and saved to Drive one fold at a time. It builds inner training-only item/difficulty/spline representations separately from the outer fold and never accesses held-out outcomes during feature fitting.

In FULL, every challenger is paired against the exact saved E04 OOF; both engineering eligibility and scientific evidence are reported. No public submission occurs.

In [ ]:
start=time.perf_counter()
RESULT=execute(
    drive_root=DRIVE,
    data_root_override=DATA_ROOT_OVERRIDE,
    phase0_root_override=str(PHASE0_ROOT),
    phase1_root_override=str(M85_ROOT),
    output_root_override=str(OUTPUT_ROOT),
    mode=MODE,
    models=tuple(EXPERIMENT_IDS),
    export_candidate=EXPORT_CANDIDATE,
    export_diagnostic_e04=EXPORT_E04_DIAGNOSTIC,
    expected_fold_hash=EXPECTED_FOLD_HASH,
)
print("Completed/reused",MODE,"in",round(time.perf_counter()-start,1),"seconds")
print("Decision:",RESULT["decision"],"winner:",RESULT["winner"])
print("Reference SHA:",RESULT["reference_oof_sha256"])
print("E04 SHA:",RESULT["e04_oof_sha256"])

## 7 · Screening reference and comparative performance

Mean pooled OOF student R² is a different estimand from mean-of-50-fold R². Both are reported, as are MAE, RMSE and school generalization.

In [ ]:
ROOT=OUTPUT_ROOT/MODE.lower()
METRICS=pd.DataFrame(RESULT["model_metrics"])
columns=[c for c in ["model","mean_fold_r2","student_pooled_r2",
    "school_pooled_r2","student_mae","student_rmse",
    "paired_fold_delta_r2","paired_student_pooled_delta_r2",
    "paired_school_delta_r2","win_fraction","corrected_t",
    "submission_eligible","science_gate"] if c in METRICS]
display(METRICS[columns].round(6))
if MODE=="SCREEN":
    print("SCREEN is engineering exploration, not confirmatory model evaluation.")
else:
    print("FULL completed:",RESULT["n_student_folds"],"student and",
          RESULT["n_school_folds"],"school-heldout folds")

## 8 · Figures: item support, instrument task administration, model gains

In [ ]:
FIGURES=ROOT/"figures"
for filename in ["01_item_support.png","02_task_admin_by_instrument.png",
                 "03_model_performance.png","04_paired_deltas.png",
                 "05_school_transfer.png"]:
    p=FIGURES/filename
    if p.is_file():
        print(filename)
        display(Image(filename=str(p)))

## 9 · Examine private item identity and complexity atlas

These tables are for internal research; do not commit child-level, school-level or scored-item records to public GitHub.

In [ ]:
TAB=ROOT/"tables"
for filename in ("diagnostic_summary.json","task_missingness_by_instrument.csv",
                 "instrument_transfer_suppressed.csv"):
    p=TAB/filename
    if p.suffix==".json":
        print(filename,json.loads(p.read_text()))
    else:
        print(filename);display(pd.read_csv(p))
print("Prompt identifiers are task+normalized expected_text, NOT temporal order.")

## 10 · Investigate nonlinear theta models and item-response gains

For B2, nonnegative hinge coefficients guarantee monotonicity in the input theta, conditional on other observed covariates. The free S3 splines and theta×instrument terms are omitted from B2.

I2 is a strongly regularized 1PL Rasch model. Item parameters are trained only on other students for each training representation; real test item parameters come from all 541 training students.

In [ ]:
selected=METRICS[METRICS.model.isin(["E04","B1","B2","B3","I1","I2","I3","I4","I5"])]
display(selected[columns].round(6))
print("Complexity metrics are orthographic proxies, not verified phoneme counts.")
print("Rasch theta and its standard error are model-derived and may differ in scale from official theta.")

## 11 · Instrument partial pooling and clip duration nuisance controls

D2 uses a common regression with penalized instrument-specific deviations. D3 uses the observed deletion-available indicator with shrinkage toward the global regression.

T1 normalizes **clip duration** by prompt and item score using training-fold statistics only. It does not claim response latency or child-specific voice activity.

In [ ]:
subset=METRICS[METRICS.model.isin(["E04","D2","D3","T1"])]
display(subset[columns].round(6))
print("Any T1 gain must be treated as a possible protocol/recording contribution.")

## 12 · FULL paired evaluation and separate decision rules

In [ ]:
if MODE=="FULL":
    COMP=pd.read_csv(TAB/"paired_comparisons.csv")
    display(COMP.round(6))
    print("Engineering candidate:",RESULT["winner"])
    print("Predeclared primary science checks:",RESULT["science_primary"])
else:
    print("No pairwise t-statistic or model promotion is valid from SCREEN.")

## 13 · Submission files and official score recording

Generated candidate CSVs are NOT uploaded automatically. The E04 diagnostic CSV is a separate opt-in output and is **not** a change to the historical HOLD_REFERENCE decision.

Once uploaded through the competition website, manually record receipt, timestamp, public R², model checksum and Git SHA. Do not tune repeatedly to the public subset.

In [ ]:
if RESULT.get("candidate_submission"):
    cand=RESULT["candidate_submission"]
    print("M8.6 candidate file:",cand["csv"])
    print("SHA256:",cand["sha256"])
    assert cand["all_test_singleton_checked"] and cand["test_permutation_checked"]
else:
    print("No candidate exported. SCREEN or engineering veto kept the output private.")
if RESULT.get("diagnostic_e04"):
    print("E04 diagnostic:",RESULT["diagnostic_e04"]["submission"]["file"]
          if "file" in RESULT["diagnostic_e04"]["submission"] else
          RESULT["diagnostic_e04"]["submission"])

## 14 · Private report and ledger

The report differentiates screening hypotheses, engineering selection, scientific evidence, and remaining limitations. Inspect null findings as well as positive ones.

In [ ]:
REPORT=ROOT/"reports"/"M86_DECISION.md"
assert REPORT.is_file()
display(Markdown(REPORT.read_text(encoding="utf-8")))
ledger=pd.read_parquet(ROOT/"ledger"/"experiments.parquet")
display(ledger[[c for c in ["model_id","status","git_sha","fold_hash",
                            "mean_fold_r2","mean_pooled_oof_r2"] if c in ledger]].tail(20))

## 15 · Phase 2 / M8.7 handoff

The research priority after M0++ is to establish genuine child speech evidence, not another broad STFT feature sweep.

Model licenses, per-checkpoint hashes, local-only inference, child response localization QA, sentence-repetition partial credit, blending/deletion near-misses, unscored-clip recovery, and recording-condition controls are the M8.7 entry criteria.

Current M8.6 **does not** perform pretrained phonetic inference or use audio, and cannot be cited as new child-speech evidence.